# **PhonePe Transaction Insights** - EDA Project

##### **Project Type** - EDA (Exploratory Data Analysis)
##### **Contribution** - Individual
##### **Domain** - Finance/Payment Systems

# **Project Summary -**

PhonePe, India's leading digital payments platform, processes billions of transactions across diverse categories including merchant payments, peer-to-peer transfers, recharge & bill payments, financial services, and insurance. This project performs a comprehensive Exploratory Data Analysis (EDA) on the PhonePe Pulse dataset spanning 2018-2024.

The dataset was extracted from the official PhonePe Pulse GitHub repository, transformed from nested JSON structures into 9 structured SQL tables covering aggregated, map (district-level), and top-performing entities across transactions, users, and insurance domains. The SQLite database contains over 1.5 million records.

Our analysis addresses 5 key business case studies: (1) Decoding Transaction Dynamics — examining how transaction volumes and values vary across payment categories, states, and quarters; (2) Device Dominance & User Engagement — understanding device brand preferences and their correlation with app usage; (3) Insurance Penetration & Growth — identifying untapped markets for insurance adoption; (4) Transaction Analysis for Market Expansion — pinpointing high-growth states and districts; (5) User Engagement & Growth Strategy — analyzing registration trends and app engagement patterns.

Through 20+ visualizations following the UBM (Univariate, Bivariate, Multivariate) framework, we uncover actionable insights such as the dominance of peer-to-peer payments in transaction value, the explosive growth of merchant payments, regional disparities in digital adoption, and the nascent but rapidly growing insurance segment. These findings inform targeted marketing strategies, feature development priorities, and market expansion decisions.

# **GitHub Link -**

https://github.com/Sanskar567/PhonePe-Transaction-Insights

# **Problem Statement**

With the increasing reliance on digital payment systems like PhonePe, understanding the dynamics of transactions, user engagement, and insurance-related data is crucial for improving services and targeting users effectively. This project aims to analyze and visualize aggregated values of payment categories, create maps for total values at state and district levels, and identify top-performing states, districts, and pin codes across transactions, users, and insurance domains.

#### **Define Your Business Objective?**

The business objective is to leverage PhonePe Pulse data to:
1. Identify transaction patterns across payment categories and regions to optimize service offerings
2. Understand device-level user engagement to improve app performance on popular devices
3. Analyze insurance penetration to identify growth opportunities in underserved markets
4. Map geographic transaction dynamics for strategic market expansion
5. Evaluate user registration and engagement trends to enhance retention strategies

# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print("Libraries imported successfully!")

### Dataset Loading

In [ ]:
# Load Dataset from SQLite database
conn = sqlite3.connect('phonepe.db')

# Load all 9 tables into DataFrames
agg_trans = pd.read_sql('SELECT * FROM aggregated_transaction', conn)
agg_user = pd.read_sql('SELECT * FROM aggregated_user', conn)
agg_ins = pd.read_sql('SELECT * FROM aggregated_insurance', conn)
map_trans = pd.read_sql('SELECT * FROM map_transaction', conn)
map_user = pd.read_sql('SELECT * FROM map_user', conn)
map_ins = pd.read_sql('SELECT * FROM map_insurance', conn)
top_trans = pd.read_sql('SELECT * FROM top_transaction', conn)
top_user = pd.read_sql('SELECT * FROM top_user', conn)
top_ins = pd.read_sql('SELECT * FROM top_insurance', conn)

conn.close()
print("All 9 tables loaded successfully!")
print(f"Tables: agg_trans, agg_user, agg_ins, map_trans, map_user, map_ins, top_trans, top_user, top_ins")

### Dataset First View

In [ ]:
# Dataset First Look - Aggregated Transactions
print("=== Aggregated Transactions (first 5 rows) ===")
display(agg_trans.head())
print("\n=== Aggregated Users (first 5 rows) ===")
display(agg_user.head())
print("\n=== Map Transactions (first 5 rows) ===")
display(map_trans.head())

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
tables = {'agg_trans': agg_trans, 'agg_user': agg_user, 'agg_ins': agg_ins,
          'map_trans': map_trans, 'map_user': map_user, 'map_ins': map_ins,
          'top_trans': top_trans, 'top_user': top_user, 'top_ins': top_ins}

for name, df in tables.items():
    print(f"{name:12s}: {df.shape[0]:>10,} rows x {df.shape[1]:>2} columns")
print(f"{'TOTAL':12s}: {sum(df.shape[0] for df in tables.values()):>10,} rows")

### Dataset Information

In [ ]:
# Dataset Info for primary tables
print("=== Aggregated Transaction Info ===")
agg_trans.info()
print("\n=== Aggregated User Info ===")
agg_user.info()
print("\n=== Map Transaction Info ===")
map_trans.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
for name, df in tables.items():
    dupes = df.duplicated().sum()
    print(f"{name:12s}: {dupes:>6} duplicates ({dupes/len(df)*100:.2f}%)")

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
for name, df in tables.items():
    nulls = df.isnull().sum().sum()
    print(f"{name:12s}: {nulls:>6} missing values")

In [ ]:
# Visualizing the missing values - focus on agg_user (has nullable brand columns)
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(agg_user.isnull(), cbar=True, yticklabels=False, cmap='YlOrRd', ax=ax)
ax.set_title('Missing Values Heatmap - Aggregated User Table')
plt.tight_layout()
plt.show()

### What did you know about your dataset?

The PhonePe Pulse dataset comprises 9 tables with over 1.5 million records spanning 2018-2024. Key observations:
- **No duplicate rows** exist in any table — data integrity is maintained.
- **Missing values** are present only in the `aggregated_user` table for `brand`, `brand_count`, and `brand_percentage` columns. This is expected as country-level aggregation does not include device-specific breakdowns.
- Data types are appropriate: numerical columns (counts, amounts) are stored as integers/floats, categorical columns (state, transaction_type) as text.
- The dataset covers 36 Indian states/UTs with quarterly granularity across 7 years.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
for name, df in tables.items():
    print(f"\n{name}: {list(df.columns)}")

In [ ]:
# Dataset Describe - Key numerical statistics
print("=== Aggregated Transaction Statistics ===")
display(agg_trans.describe())
print("\n=== Map Transaction Statistics ===")
display(map_trans.describe())

### Variables Description

**Aggregated Transaction:** `state` (region), `year`/`quarter` (time period), `transaction_type` (payment category), `transaction_count`/`transaction_amount` (volume and value)

**Aggregated User:** `registered_users` (total registrations), `app_opens` (engagement metric), `brand`/`brand_count`/`brand_percentage` (device distribution)

**Map Tables:** District-level breakdown of transactions, users, and insurance

**Top Tables:** Top 10 states, districts, and pincodes by each metric with `entity_name`, `entity_type` classification

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for key categorical variables
print("Unique States (agg_trans):", agg_trans['state'].nunique())
print("Unique Years:", sorted(agg_trans['year'].unique()))
print("Unique Quarters:", sorted(agg_trans['quarter'].unique()))
print("Transaction Types:", agg_trans['transaction_type'].unique().tolist())
print("\nDevice Brands:", agg_user['brand'].dropna().unique().tolist())
print("\nTop entity types:", top_trans['entity_type'].unique().tolist())

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Data Wrangling: Clean and prepare datasets for analysis

# 1. Standardize state names (title case, replace hyphens)
for df in [agg_trans, agg_user, agg_ins, map_trans, map_user]:
    if 'state' in df.columns:
        df['state'] = df['state'].str.replace('-', ' ').str.replace('&', 'and').str.title()

# 2. Create a year-quarter combined column for time-series analysis
for df in [agg_trans, agg_user, agg_ins, map_trans, map_user]:
    if 'year' in df.columns and 'quarter' in df.columns:
        df['year_quarter'] = df['year'].astype(str) + '-Q' + df['quarter'].astype(str)

# 3. Filter out country-level aggregate rows for state analysis
agg_trans_states = agg_trans[agg_trans['state'] != 'India'].copy()
agg_user_states = agg_user[agg_user['state'] != 'India'].copy()
agg_ins_states = agg_ins[agg_ins['state'] != 'India'].copy()

# 4. Create transaction amount in crores for readability
agg_trans_states['amount_crores'] = agg_trans_states['transaction_amount'] / 1e7
map_trans['amount_crores'] = map_trans['transaction_amount'] / 1e7

# 5. Fill missing brand values
agg_user_states_brands = agg_user_states[agg_user_states['brand'].notna()].copy()

print("Data wrangling complete!")
print(f"State-level transactions: {len(agg_trans_states)} rows")
print(f"State-level users: {len(agg_user_states)} rows")
print(f"State-level insurance: {len(agg_ins_states)} rows")
print(f"Users with brand data: {len(agg_user_states_brands)} rows")

### What all manipulations have you done and insights you found?

1. **State name standardization**: Replaced hyphens and ampersands for consistency (e.g., 'andhra-pradesh' → 'Andhra Pradesh')
2. **Year-Quarter column**: Created a combined temporal key for time-series visualization
3. **Separated country vs state data**: Filtered out 'India' aggregate rows to avoid double-counting in state-level analysis
4. **Amount scaling**: Converted raw amounts to Crores (÷10⁷) for readability in charts
5. **Brand data isolation**: Separated rows with valid device brand data for device analysis

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Chart - 1: Distribution of Transaction Types by Count
df_type = agg_trans_states.groupby('transaction_type')['transaction_count'].sum().sort_values(ascending=False).reset_index()
sns.barplot(data=df_type, x='transaction_count', y='transaction_type')
plt.title('Total Transaction Count by Category (2018-2024)')
plt.xlabel('Count (Billions)')
plt.show()

##### 1. Why did you pick the specific chart?

To understand which payment categories are most frequently used by customers.

##### 2. What is/are the insight(s) found from the chart?

Merchant payments and Peer-to-peer payments dominate the volume, reflecting high daily utility usage.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

High frequency in merchant payments suggests a strong ecosystem for retail digitisation. Focus on micro-transaction features.

#### Chart - 2

In [ ]:
# Chart - 2: Distribution of Transaction Types by Amount
df_amt = agg_trans_states.groupby('transaction_type')['amount_crores'].sum().sort_values(ascending=False).reset_index()
sns.barplot(data=df_amt, x='amount_crores', y='transaction_type')
plt.title('Total Transaction Amount by Category in Crores (2018-2024)')
plt.show()

##### 1. Why did you pick the specific chart?

To contrast the value contribution of different categories against their volume.

##### 2. What is/are the insight(s) found from the chart?

While Merchant payments are high in count, Peer-to-peer payments contribute significantly more in total value (INR).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

P2P is the value driver; Merchant is the engagement driver. Optimize P2P security and Merchant settlement speed.

#### Chart - 3

In [ ]:
# Chart - 3: Multi-Year Growth Trend (Transaction Amount)
df_yearly = agg_trans_states.groupby('year')['amount_crores'].sum().reset_index()
sns.lineplot(data=df_yearly, x='year', y='amount_crores', marker='o', linewidth=3)
plt.title('Year-on-Year Growth in Total Transaction Value')
plt.ylabel('Amount in Crores')
plt.show()

##### 1. Why did you pick the specific chart?

To track the overall growth trajectory of the platform.

##### 2. What is/are the insight(s) found from the chart?

Exponential growth is visible from 2020 onwards, likely accelerated by the COVID-19 pandemic and UPI adoption.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

The market is still in a high-growth phase. Scaling infrastructure is critical to handle future demand.

#### Chart - 4

In [ ]:
# Chart - 4: Top 10 States by Transaction Value
top_10_states = agg_trans_states.groupby('state')['amount_crores'].sum().sort_values(ascending=False).head(10).reset_index()
sns.barplot(data=top_10_states, x='amount_crores', y='state')
plt.title('Top 10 States by Cumulative Transaction Value')
plt.show()

##### 1. Why did you pick the specific chart?

To identify the key geographical markets contributing to revenue.

##### 2. What is/are the insight(s) found from the chart?

Karnataka, Maharashtra, and Telangana lead the charts, indicating high digital penetration in southern and western India.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Focus marketing efforts and localized features in these 'Tier 1' digital states while identifying why others lag.

#### Chart - 5

In [ ]:
# Chart - 5: Market Share of Device Brands
brand_dist = agg_user_states_brands.groupby('brand')['brand_count'].sum().sort_values(ascending=False).head(10).reset_index()
plt.pie(brand_dist['brand_count'], labels=brand_dist['brand'], autopct='%1.1f%%', startangle=140, explode=[0.1]+[0]*9)
plt.title('Engagement Share by Device Brand')
plt.show()

##### 1. Why did you pick the specific chart?

To understand the hardware ecosystem used by PhonePe customers.

##### 2. What is/are the insight(s) found from the chart?

Xiaomi and Vivo users make up a massive portion of the user base, followed by Samsung.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

App optimization should be prioritized for highly-segmented Android brands (Xiaomi/Vivo) to ensure high retention.

#### Chart - 6

In [ ]:
# Chart - 6: Average Transaction Value (ATV) by State
# Calculate ATV: Total Amount / Total Count
state_atv = agg_trans_states.groupby('state').agg({'transaction_amount':'sum', 'transaction_count':'sum'})
state_atv['atv'] = state_atv['transaction_amount'] / state_atv['transaction_count']
top_atv = state_atv['atv'].sort_values(ascending=False).head(10).reset_index()
sns.barplot(data=top_atv, x='atv', y='state')
plt.title('Top 10 States by Average Transaction Value')
plt.show()

##### 1. Why did you pick the specific chart?

To see where high-ticket transactions are occurring on average.

##### 2. What is/are the insight(s) found from the chart?

Certain smaller states/territories show higher ATV, potentially due to fewer but larger institutional transfers.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Higher ticket sizes in specific regions can be targeted with premium financial products like high-limit credit lines.

#### Chart - 7

In [ ]:
# Chart - 7: Quarterly Transaction Volume Seasonality
q_vol = agg_trans_states.groupby('quarter')['transaction_count'].mean().reset_index()
sns.barplot(data=q_vol, x='quarter', y='transaction_count')
plt.title('Average Transaction Volume by Quarter')
plt.show()

##### 1. Why did you pick the specific chart?

To detect seasonal peaks in digital payments.

##### 2. What is/are the insight(s) found from the chart?

Q4 (Oct-Dec) often shows peak volume, likely coinciding with festive seasons and sales in India.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Prepare server capacity for Q4; launch festive-specific cashback and merchant offers in October.

#### Chart - 8

In [ ]:
# Chart - 8: User Engagement (App Opens vs Registered Users)
sns.scatterplot(data=agg_user_states.groupby(['state', 'year']).sum().reset_index(), 
                x='registered_users', y='app_opens', hue='state', legend=False)
plt.title('Correlation: App Opens vs Registered Users by State')
plt.show()

##### 1. Why did you pick the specific chart?

To evaluate if a higher user base leads to proportional engagement.

##### 2. What is/are the insight(s) found from the chart?

There is a strong linear correlation, but some states show higher engagement (app opens) per user than others.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

States with high users but low app opens need re-engagement campaigns to improve daily active usage.

#### Chart - 9

In [ ]:
# Chart - 9: Insurance Segment Growth
ins_growth = agg_ins_states.groupby('year')['transaction_count'].sum().reset_index()
sns.lineplot(data=ins_growth, x='year', y='transaction_count', marker='s', color='red')
plt.title('Growth in Insurance Policy Purchases')
plt.show()

##### 1. Why did you pick the specific chart?

To track the adoption of the relatively new insurance product line.

##### 2. What is/are the insight(s) found from the chart?

Insurance shows a sharp upward trend, indicating increasing trust in digital insurance purchases.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Expand insurance coverage types (Health, Life, Asset) as users show readiness for digital insurance.

#### Chart - 10

In [ ]:
# Chart - 10: Top 15 Districts by Transaction Amount
top_districts = map_trans.groupby('district')['amount_crores'].sum().sort_values(ascending=False).head(15).reset_index()
sns.barplot(data=top_districts, x='amount_crores', y='district')
plt.title('Top 15 Districts by Cumulative Transaction Value')
plt.show()

##### 1. Why did you pick the specific chart?

To drill down from state-level to district-level performance.

##### 2. What is/are the insight(s) found from the chart?

Metropolitan districts like Bengaluru Urban, Pune, and Hyderabad are the engines of UPI growth.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Physical merchant acquisition (QR codes) should remain aggressive in these districts while expanding to neighboring districts.

#### Chart - 11 - Correlation Heatmap

In [ ]:
# Chart - 11: Correlation Heatmap between Numerical Variables
corr_df = agg_trans_states[['year', 'quarter', 'transaction_count', 'transaction_amount', 'amount_crores']].corr()
sns.heatmap(corr_df, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Analysis of Transaction Metrics')
plt.show()

Correlation shows strong relationship between year and amount, confirming growth over time.

#### Chart - 12 - Pair Plot

In [ ]:
# Chart - 12: Pair Plot for Transaction Trends
sns.pairplot(agg_trans_states[['transaction_count', 'amount_crores', 'year']], hue='year', palette='viridis')
plt.show()

The pair plot visualizes the tightening correlation between count and amount as time progresses.

#### Chart - 13

In [ ]:
# Chart 13: Category Share Transition Over Years
pivot_df = agg_trans_states.groupby(['year', 'transaction_type'])['amount_crores'].sum().unstack()
pivot_df.plot(kind='area', stacked=True, figsize=(12, 6))
plt.title('Transaction Value: Category Share Transition (2018-2024)')
plt.ylabel('Amount in Crores')
plt.show()

##### 1. Why did you pick the specific chart?

To see how the mix of transactions has evolved.

##### 2. What is/are the insight(s) found from the chart?

Merchant payments are growing as a percentage of total value compared to earlier years.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

The platform is maturing from a P2P tool to a comprehensive commercial ecosystem.

#### Chart - 14

In [ ]:
# Chart 14: Quarterly User Base Expansion
user_growth = agg_user_states.groupby('year_quarter')['registered_users'].sum().reset_index()
sns.barplot(data=user_growth.tail(12), x='year_quarter', y='registered_users')
plt.xticks(rotation=45)
plt.title('User Base Expansion (Last 12 Quarters)')
plt.show()

##### 1. Why did you pick the specific chart?

To monitor recent user acquisition velocity.

##### 2. What is/are the insight(s) found from the chart?

Steady linear growth in users continues even in 2023-24.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Consistent growth suggests the digital divide is narrowing at a predictable pace.

#### Chart - 15

In [ ]:
# Chart 15: Top Districts for User Registration
top_reg_dist = map_user.groupby('district')['registered_users'].max().sort_values(ascending=False).head(10).reset_index()
sns.barplot(data=top_reg_dist, x='registered_users', y='district', palette='rocket')
plt.title('Top 10 Districts by Registered Users')
plt.show()

##### 1. Why did you pick the specific chart?

To identify the largest user hubs.

##### 2. What is/are the insight(s) found from the chart?

Bengaluru Urban remains the single largest hub by registered users.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Marketing campaigns can be hyper-localised for top districts to drive specific feature adoption (like Insurance).

#### Chart - 16

In [ ]:
# Chart 16: Insurance vs Financial Services Growth
f1 = agg_trans_states[agg_trans_states['transaction_type'] == 'Financial Services'].groupby('year')['transaction_count'].sum()
f2 = agg_ins_states.groupby('year')['transaction_count'].sum()
merged_fs = pd.concat([f1, f2], axis=1, keys=['Financial Services', 'Insurance']).fillna(0)
merged_fs.plot(marker='o')
plt.title('Comparative Growth: Financial Services vs Insurance')
plt.show()

##### 1. Why did you pick the specific chart?

To compare adoption speeds of different financial segments.

##### 2. What is/are the insight(s) found from the chart?

Insurance is showing faster recent growth than general financial service micro-payments.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Prioritize insurance-linked payment reminders and cross-selling.

#### Chart - 17

In [ ]:
# Chart 17: User Base by State (Log Scale to show disparity)
state_users = agg_user_states.groupby('state')['registered_users'].max().sort_values(ascending=False).reset_index()
sns.barplot(data=state_users, x='registered_users', y='state')
plt.xscale('log')
plt.title('State-wise Registered Users (Log Scale)')
plt.show()

##### 1. Why did you pick the specific chart?

To visualize the reach across all states including smaller ones.

##### 2. What is/are the insight(s) found from the chart?

Huge disparity between top and bottom states, spanning several orders of magnitude.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Untapped potential in lower-ranked states requires localized trust-building campaigns.

#### Chart - 18

In [ ]:
# Chart 18: Median Transaction Amount Over Years
agg_trans_states['avg_ticket_size'] = agg_trans_states['transaction_amount'] / agg_trans_states['transaction_count']
sns.boxenplot(data=agg_trans_states, x='year', y='avg_ticket_size')
plt.ylim(0, 5000)
plt.title('Distribution of Average Ticket Size per Transaction')
plt.show()

##### 1. Why did you pick the specific chart?

To analyze if transaction sizes are increasing or decreasing.

##### 2. What is/are the insight(s) found from the chart?

Ticket sizes are stabilizing, implying UPI is being used for daily mundane expenses (coffee, groceries).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

The platform has become an essential utility for daily low-value payments.

#### Chart - 19

In [ ]:
# Chart 19: App Opens to Transcation Volume Ratio
merged_df = agg_user_states.merge(agg_trans_states.groupby(['state', 'year', 'quarter']).sum().reset_index(), on=['state', 'year', 'quarter'])
merged_df['engagement_ratio'] = merged_df['app_opens'] / merged_df['transaction_count']
sns.histplot(merged_df['engagement_ratio'], bins=30, kde=True)
plt.title('Distribution of App Opens per Transaction')
plt.show()

##### 1. Why did you pick the specific chart?

To see how many times a user opens the app for every one transaction.

##### 2. What is/are the insight(s) found from the chart?

The ratio shows users open the app many times, likely for balance checks, bill status, etc.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

The app serves as a 'hub' beyond just payments. Monetize the 'app open' with relevant ads/offers.

#### Chart - 20

In [ ]:
# Chart 20: Top 10 Pincodes by Transaction Amount
top_pins = top_trans[top_trans['entity_type'] == 'pincode'].groupby('entity_name')['transaction_amount'].sum().sort_values(ascending=False).head(10).reset_index()
sns.barplot(data=top_pins, x='transaction_amount', y='entity_name')
plt.title('Top 10 Pincodes by Transaction Value')
plt.show()

##### 1. Why did you pick the specific chart?

Hyper-local performance analysis.

##### 2. What is/are the insight(s) found from the chart?

Certain PIN codes in urban tech-hubs are hotspots for transactions.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Targeted offline activation and merchant tie-ups in these high-value zones.

## **5. Solution to Business Objective**

#### What do you suggest the client to achieve Business Objective ?
Explain Briefly.

To achieve the business objectives, we suggest the client focus on:
1. **Deepen penetration in 'Tier 2' and 'Tier 3' states**: While top states like Karnataka and Maharashtra are well-penetrated, addressing the gap in lower-ranked states like Bihar or Odisha with localized marketing campaigns can unlock the next wave of growth.
2. **Optimize app performance for popular device brands**: Ensuring the app remains responsive and lightweight on Android brands like Xiaomi, Vivo, and Samsung is critical as they encompass the majority of the user base.
3. **Scaling Insurance and Financial Services**: The rapid growth in insurance purchases indicates user trust. The company should expand its insurance and financial service portfolio to turn PhonePe into a comprehensive financial services hub.
4. **Hyper-local merchant expansion**: Focus on acquiring merchants in high-performing pincodes identified in the analysis to maximize transaction volume.
5. **Incentivize high-value P2P and recurring payments**: P2P remains a major value driver; introducing secure, high-value transfer features can retain high-ticket users.

# **Conclusion**

The Exploratory Data Analysis of the PhonePe Pulse dataset reveal significant trends in the Indian digital payment landscape:
- **Exponential Growth**: Transaction volumes and values have seen sustained, exponential growth, reflecting a shift towards a cashless economy.
- **Dominant Categories**: While micro-transactions are driven by Merchant payments, large-scale value transfers are dominated by Peer-to-Peer payments.
- **Geographic Disparity**: Digital payment adoption is currently concentrated in urban tech-hubs and southern/western states, presenting a clear opportunity for expansion into northern and eastern regions.
- **User Engagement**: There is a strong, growing base of registered users, and users frequency open the app for several daily utility needs beyond just payments.
- **Emerging Segments**: The insurance sector is showing promising early growth, indicating the success of product diversification.

Overall, PhonePe is well-positioned as a market leader, but continued success will depend on expanding into underserved regions and scaling high-trust financial services like insurance.

### ***Hurrah! You have successfully completed your EDA Capstone Project !!!***